# Feature Extraction for RNN
### Importing necessary libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer
sns.set()
# Access data using Google BigQuery.
from google.colab import auth
from google.cloud import bigquery
from google.colab import drive
drive.mount('/content/gdrive')
from google.colab import files

In [ ]:
auth.authenticate_user()
print('Authenticated')


### Query to obtain features

In [ ]:
project_id='eicudata'

In [ ]:
%%bigquery --project $project_id rawdf
select b.uniquepid, b.patienthealthsystemstayid,b.patientunitstayid,gender,age,ethnicity,apacheadmissiondx,admissionheight,admissionweight, eyes, motor, verbal, (eyes+motor+verbal) as GCSTotal, temperature,respiratoryrate,heartrate,glucose,(select min(labresult)
from `physionet-data.eicu_crd.lab`
where patientunitstayid = a.patientunitstayid and labname like '%O2 Sat%'
group by labname) as O2,fio2,ph,hospitaldischargestatus,unitdischargestatus,(unitdischargeoffset-hospitaladmitoffset)/60/24 as ICUstay
from `physionet-data.eicu_crd.apacheapsvar` a
join `physionet-data.eicu_crd.patient` b ON (
a.patientunitstayid =b.patientunitstayid
)
order by uniquepid;

In [ ]:
rawdf

### We will consider patients that have had multiple hospitalizations -

In [ ]:
rawdf = rawdf[rawdf.groupby('patienthealthsystemstayid').patienthealthsystemstayid.transform(len) > 1]
rawdf

### Query to obtain some vital features

In [ ]:
%%bigquery --project $project_id vital
select patientunitstayid, avg(systemicsystolic) as SystolicBP, avg(systemicdiastolic) as DiastolicBP, avg((2*systemicdiastolic + systemicsystolic)/3) as MAP
from `physionet-data.eicu_crd.vitalperiodic` 
where systemicsystolic & systemicdiastolic is not null
group by patientunitstayid
order by patientunitstayid

In [ ]:
vital

### Merging all the features together

In [ ]:
newrawdf = pd.merge(left=rawdf,right=vital,how='left')

In [ ]:
newrawdf

### Checking for null values - 

In [ ]:
newrawdf.info()

In [ ]:
newrawdf.describe(include='all')

### Imputing height and weight missing values 

In [ ]:
newrawdf['admissionheight']=newrawdf.admissionheight.fillna(newrawdf.admissionheight.median())
newrawdf['admissionweight']=newrawdf.admissionweight.fillna(newrawdf.admissionweight.median())

### Converting age column to numeric datatype

In [ ]:
newrawdf['age'].replace({'> 89': '90'}, inplace=True)
newrawdf['age']=newrawdf['age'].apply(pd.to_numeric)

In [ ]:
newrawdf.info()

### Removing missing values for gender,ethnicity and hospital discharge status

In [ ]:
missing = newrawdf[(newrawdf['gender'] == '') |(newrawdf['gender'] == 'Other')|(newrawdf['gender'] == 'Unknown')| (newrawdf['ethnicity'] == '')|(newrawdf['hospitaldischargestatus']=='')].index
newrawdf = newrawdf.drop(index=missing)
newrawdf = newrawdf.dropna()

In [ ]:
newrawdf.isnull().sum()

In [ ]:
newrawdf

In [ ]:
#ewrawdf['apacheadmissiondx'].unique()